## 一、基础概念
### 1.1 LangGraph定义
  - LangGraph 是用于构建 有状态、多步骤 LLM应用的框架，本质是将应用建模为 有向图，其中节点执行计算，边定义流程。  
  - LangGrph 的核心特点包括：状态化、多步骤、可记忆、可中断、可恢复。

### 1.2 核心组件
  - **StateGraph** 是 LangGraph 中最核心的抽象之一，用于构建**带状态流转的任务图**，其核心组件如下：  
  - **状态（State）**：全局共享的数据结构（通常是TypedDict / Pydantic / dataclass），在所有节点之间流动，用于记录所有关键信息，相当于AI的“短期记忆”。  
  - **节点（Node）**：每个节点表示一个计算单元（特定功能函数 / LLM 调用 / 工具调用等），且每个节点都可以读取/更新 State。  
    -  **入口/出口节点（Entry & Finish）**：  
    入口：START（内置特殊节点）/set_entry_point("xxx")函数来指定；  
    终止：END（内置特殊节点）  
  - **边（Edge）**：每个边是python函数，定义节点间的执行顺序。包括「普通边（Normal Edge）、条件边（Condition Edge）即根据条件动态选择下一个节点」。  
  - **中断点（Checkpointer）**：中断点是节点执行过程中，可以保存当前状态，并中断执行，等待下一次执行时恢复状态。  
  **StateGraph** 本质是一个“可控状态流 + 有向无环图(DAG)/有条件流程 + 节点函数执行”的组合框架。  




## 二、环境安装

In [ ]:
# 安装基础包
! pip install langgraph langchain langchain-openai

In [ ]:
# 安装可选依赖
! pip install langchain-anthropic 
! pip install langchain-supervisor 
! pip install langchain-community
! pip install graphviz

## 三、基础核心组件
### 3.1 最小结构一览

In [ ]:

from langgraph.graph import StateGraph, END

# 示例：定义一个简单图
graph = StateGraph(dict)

def node_a(state):
    return {"value": "from A"}

def node_b(state):
    return {"value": "from B"}

graph.add_node("A", node_a)
graph.add_node("B", node_b)

graph.set_entry_point("A")  # 指定「自定节点」为入口
graph.add_edge("A", "B")
graph.add_edge("B", END)

app = graph.compile()

# 打印并可视化 Mermaid
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png()) # 或者使用 draw_png 如果配置了 pygraphviz

### 3.2 State 实现
  - 全局共享的数据结构（通常是TypedDict / Pydantic / dataclass），在所有节点之间流动，用于记录所有关键信息，相当于AI的“短期记忆”。 

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# 自定义 State 结构
class State(TypedDict):
    value: str      # 后续Node中可「覆盖」更新
    step: int

class AgentState(TypedDict):
    messages: Annotated[list, operator.add] # 后续Node中可「累加」更新，在这里起到多轮聊天的作用
    current_step: int


### 3.3 Node 实现
  - 每个节点表示一个计算单元（特定功能函数 / LLM 调用 / 工具调用等），且每个节点都可以读取/更新 State。

In [ ]:
def my_node(state: AgentState) -> dict:
    """
    节点函数必须：
    1. 接收 state 作为参数
    2. 返回字典，用于更新 state
    """
    messages = state["messages"]

    # Debug: 查看输入
    print(f"输入消息数量: {len(messages)}")
    
    # 执行某些操作，例如调用 LLM
    try:
        ai_response = llm.invoke(messages)
    except Exception as e:
        print(f"LLM 调用失败: {e}")
        raise

    # Debug: 查看输出
    print(f"LLM 输出: {ai_response}")
    
    # 返回更新状态，这里LangGraph内置函数会将 LLM 的响应累加到消息列表中
    return {
        "messages": [ai_response]
    }


### 3.3 Edge 实现
  - 每个边是python函数，定义节点间的执行顺序。包括「普通边（Normal Edge）、条件边（Condition Edge）即根据条件动态选择下一个节点」。  

In [ ]:
from langgraph.graph import StateGraph, END

# 1. 定义节点函数
def node1_func(state: AgentState):
    print(f"👉 node1: step = {state['current_step']}")
    return {
        "messages": ["node1_message"],
        "current_step": state["current_step"] + 1
    }

def node2_func(state: AgentState):
    print(f"👉 node2: step = {state['current_step']}")
    return {
        "messages": ["node2_message"]
    }

# 2. 构建 Graph
graph = StateGraph(AgentState)

graph.add_node("node1", node1_func)
graph.add_node("node2", node2_func)

# ======================
# 3. 普通边：总是从 node1 到 node2
# ======================
graph.add_edge("node1", "node2")

# ======================
# 4. 条件边：根据路由条件决定流向
# ======================
def router(state: AgentState) -> bool:
    # 大于5则结束，否则继续循环
    if state["current_step"] > 5:
        return False
    else:
        return True

graph.add_conditional_edges(
    "node2",                    # 给 node2 加“分支路由逻辑”
    router,                     # 根据当前 state，返回一个“路由标签”
    {True: "node1", False: END} # 根据路由标签，决定“分支路由”的方向
)

# 5. 设置入口
graph.set_entry_point("node1")
# 6. 编译与运行
app = graph.compile()
result = app.invoke({
    "messages": [],
    "current_step": 0
})

print("\n最终结果：", result)
# 7. 打印并可视化 Mermaid
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

### 3.4 完整小示例实现
#### 3.4.1 简单对话机器人（可运行）
  - 用 LangGraph 维护消息列表作为状态，串起“用户输入 → 模型回复” 的复杂路线，让对话有上下文、能连贯。

##### 3.4.1.1 非流式输出的简单对话机器人

In [ ]:
import os
from dotenv import load_dotenv

from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

# 1. 加载.env 文件中的环境变量
load_dotenv()

# 2. 创建 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",   # 自选模型
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 3. 定义状态
class AgentState(TypedDict):
    # Annotated + add_messages 共同实现「对话历史自动累加」
    messages: Annotated[list, add_messages]
    current_step: int

# 4. 定义节点函数
def chatbot(state: AgentState) -> dict:
    ai_response = llm.invoke(state["messages"])
    
    # 返回更新状态，这里LangGraph内置函数会将 LLM 的响应累加到消息列表中
    return {
        "messages": [ai_response]
    }

# 5. 构建图
graph = StateGraph(AgentState)

graph.add_node("chatbot", chatbot) # 流程图中注册节点 chatbot
graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

# 6. 打印并可视化 Mermaid
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

# 7. 运行 chatbot
from langchain_core.messages import HumanMessage

ai_response = app.invoke({
    "messages": [HumanMessage(content="你好，请介绍一下 LangGraph")],
})

print("\n完整消息列表：", ai_response["messages"])
print("\n最终回复：", ai_response["messages"][-1].content)

##### 3.4.1.2 流式输出的简单对话机器人
  - 下面是 **错误🙅代码展示（捣鼓了一上午，思路仍是一片混乱的代码）**：

In [ ]:
import os
from dotenv import load_dotenv

from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
# from langchain_community.chat_models.tongyi import ChatTongyi

from langchain_core.messages import HumanMessage, AIMessageChunk, AIMessage


# 1. 加载.env 文件中的环境变量
load_dotenv()

# 2. 创建 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",   # 自选模型
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,             # 启用流式输出，便于LangGraph流式交互
)

# 3. 定义状态
class AgentState(TypedDict):
    # Annotated + add_messages 共同实现「对话历史自动累加」
    messages: Annotated[list, add_messages]

# ======================
# 4. 定义节点（关键：流式需使用 stream + yield）
# ======================
def chatbot(state: AgentState):
    """流式节点：逐 token 产出 AIMessageChunk（供 LangGraph 外层消费），同时最终返回完整消息"""
    # full_content = llm.invoke(state["messages"]) # 同步调用，等待完整结果

    """LangGraph 官方标准流式节点：直接 yield 模型返回的 chunk"""
    for chunk in llm.stream(state["messages"]):
        yield {
            "messages": [AIMessageChunk(content=chunk.content)]
        }


# 5. 构建图
graph = StateGraph(AgentState)

graph.add_node("chatbot", chatbot) # 流程图中注册节点 chatbot
graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

# 6. 打印并可视化 Mermaid
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

# 7. 运行 chatbot
from langchain_core.messages import HumanMessage
## 7.1 invoke 同步阻塞，等待完整结果
def non_stream_graph_updates(user_input: str):
    # 同步阻塞，等待完整结果
    ai_response = app.invoke({
        "messages": [HumanMessage(content=user_input)],
        })
    # 打印完整回复
    print("\nAssistant:", ai_response["messages"][-1].content)

## 7.2 streaming 流式输出，返回的是「事件迭代器」，实时输出流程执行中的状态更新
def stream_graph_updates(user_input: str):
    """接收 LangGraph 的消息流，并逐 token 打印输出"""
    print("Assistant:", end=" ", flush=True)

    for event in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="updates",   # 👈 推荐用 updates（更通用）
    ):
        # event -> {node_name: node_output}，遍历事件中的所有节点输出（当前流程只有 chatbot 节点）
        for node_output in event.values():
            messages = node_output.get("messages", [])
            for chunk in messages:
                # print(type(chunk))  # 调试输出，查看接收到的 chunk 结构
                # ⭐ 只打印“增量”，避免最终完整重复
                if isinstance(chunk, AIMessageChunk) and chunk.content:
                    print(chunk.content, end="", flush=True)

    print() # 换行

# 8. 测试
# while True:
#     user_input = input("User: ")
#     if user_input.lower() in ["exit", "quit", "q"]:
#         print("Goodbye!")
#         break

#     # non_stream_graph_updates(user_input)
#     stream_graph_updates(user_input)
stream_graph_updates("你好，给我介绍一下 LangGraph 吧")

  **运行上面👆代码后，会发现无任何流式输出？**  

  - **分析问题与解决方法**:
    1. 节点定义中用 yield 做 generator，但 stream_mode="updates" 会把所有 yield 累积后一次性发出，不是逐 token。  
      - *解决方法：节点改为普通函数，用 llm.invoke() 返回完整结果 —— LangGraph 的流式机制在图的外层处理，节点本身不需要 yield。*  
    2. add_messages reducer 会把 AIMessageChunk 转成 AIMessage，导致 isinstance(chunk, AIMessageChunk) 永远为 False，什么都打印不出来。
    3. stream_mode="updates" 的用法，这是节点级更新，节点跑完才输出，因此也不是 token 级流式。
      - *解决方法：改为 stream_mode="messages" 的用法，这才是 LangGraph 官方推荐的 token 级流式方案 —— LangGraph 自动拦截 LLM 内部的 streaming，逐 token 推送 AIMessageChunk。*
  
  - **拓展**：
    - LangGraph 的“流式责任分层”是 「**节点只负责“算结果”，流式由 LangGraph 外层负责**」 

      👉 **节点不做流式**
      ```python
      def chatbot(state):
          response = llm.invoke(state["messages"])
          return {"messages": [response]}
      ```

      👉 流式交给：
      ```python
      app.stream(..., stream_mode="messages")
      ```

      ✅ LangGraph 内部会自动做这件事：
        1. （同步）调用节点
        2. 发现里面用了 LLM（invoke）
        3. 👉 **内部拦截 LLM 的 streaming**
        4. 把 token 一个个转成：AIMessageChunk
        5. 实时往外推  

    > 一句话总结：👉 **流式的位置应该在“图外”，而不是“节点里”，LangGraph 的流式 = “框架级能力”，不是“节点级能力”**  

##### 3.4.1.2 流式输出的简单对话机器人
  - 下面是 **正确🙆代码展示**：

In [ ]:
import os
from dotenv import load_dotenv

from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage, AIMessageChunk, AIMessage


# 1. 加载.env 文件中的环境变量
load_dotenv()

# 2. 创建 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",   # 自选模型
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,             # 启用流式输出，便于LangGraph流式交互
)

# 3. 定义状态
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ======================
# 4. 定义节点（普通函数，不用 generator）
# ======================
def chatbot(state: AgentState):
    """普通节点：调用 LLM 返回完整结果。
    token 级流式由 LangGraph 的 stream_mode="messages" 在外层自动处理，
    节点本身不需要 yield。"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


# 5. 构建图
graph = StateGraph(AgentState)
graph.add_node("chatbot", chatbot)
graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()


# ====================== 运行方式 ======================
# 方式1: invoke 同步阻塞
def non_stream_graph_updates(user_input: str):
    ai_response = app.invoke({
        "messages": [HumanMessage(content=user_input)],
    })
    print("\nAssistant:", ai_response["messages"][-1].content)


# 方式2: stream_mode="updates" — 节点级更新（每个节点完成后输出一次）
def stream_graph_updates(user_input: str):
    """updates 模式：每个节点执行完毕后发出一次完整结果，适合观察图的执行流程。
    注意：这不是 token 级流式，而是节点级流式。"""
    for event in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="updates",
    ):
        for node_name, node_output in event.items():
            messages = node_output.get("messages", [])
            for msg in messages:
                if isinstance(msg, (AIMessage, AIMessageChunk)) and msg.content:
                    print(f"Assistant [{node_name}]: {msg.content}")


# 方式3: stream_mode="messages" — 真正的 token 级流式输出 ⭐
def stream_graph_messages(user_input: str):
    """messages 模式：LangGraph 自动拦截 LLM 内部的流式调用，
    逐 token 推送 AIMessageChunk，实现打字机效果。"""
    print("Assistant:", end=" ", flush=True)

    for msg, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="messages",
    ):
        # msg 是 AIMessageChunk（增量）或 AIMessage（完整，节点结束时）
        # 只打印增量 chunk，跳过最终完整消息避免重复
        if isinstance(msg, AIMessageChunk) and msg.content:
            print(msg.content, end="", flush=True)

    print()  # 换行


# ====================== 测试 ======================
# print("=== stream_mode='updates' (节点级) ===")
# stream_graph_updates("你好，给我介绍一下 LangGraph 吧")

print("\n=== stream_mode='messages' (token级流式) ===")
stream_graph_messages("你好，给我介绍一下 LangGraph 吧")

**概念要点注意：**
  1. **Annotated[类型, 元数据(或规则)]**：Annotated给字段附加额外规则，这是 Python 3.9+ 引入的标准库工具（从typing导入），作用是「给类型添加元数据/规则」。  
     - 语法：Annotated[基础类型, 规则1, 规则2, ...]
     - 「规则」会被 LangGraph 识别并生效
  2. **add_messages**：是 LangGraph 内置定义「消息追加处理器」函数。
     - add_messages 是从 langgraph.graph 导入的核心工具（需显式导入：from langgraph.graph import add_messages）
     - 核心作用：自动累加合并「节点返回的新消息」和「原有状态中的旧消息」  
  3. **stream_mode**：是 LangGraph 内置定义「流式模式」函数。功能对比如下👇：  

| stream_mode  | 返回结构                         | 粒度        | 是否包含节点信息      | 是否支持 token 流 | 典型用途            |
| ------------ | ---------------------------- | --------- | ------------- | ------------ | --------------- |
| `"values"`   | `state`（完整状态）                | 粗         | ❌             | ❌            | 调试 / 查看最终状态     |
| `"updates"`  | `{node_name: partial_state}` | 中         | ✅             | ❌（默认）        | 多节点流程、Agent编排   |
| `"messages"` | `(chunk, metadata)`          | 细（token级） | ⚠️（metadata中） | ✅            | ChatGPT式流式输出    |
| `"events"`   | 事件流（内部执行事件）                  | 最细        | ✅             | ⚠️（间接）       | Debug / tracing |
| `"debug"`    | 超详细执行信息                      | 最细        | ✅             | ❌            | 深度调试（开发框架级）     |


#### 3.4.2 带工具调用的机器人
  - 带工具调用的 StateGraph 的核心逻辑：构建***思考 -> 判断 -> 执行 -> 再思考*** 的闭环：  
    > 入口 → chat_bot（思考）→ tool_router（判断）→  
    > ① 无工具调用 → 结束  
    > ② 有工具调用 → tools（执行）→ chat_bot（基于工具结果生成回复）→ 结束  

  - 工具调用的核心5步：  

  | 步骤 | 组件                 | 干啥用                    |
| -- | ------------------ | ---------------------- |
| 1.定义工具 | @tool        | 把普通函数包装成 LangGraph 能识别的工具       |
| 2.绑定工具  | bind_tools() | 告诉模型：“你现在有这些技能”         |
| 3.判断是否调用  | tool_router | 看 messages 中 **tool_calls** 是否为空，决定走哪条路 |
| 4.执行工具  | ToolNode     | **内置预构建节点**：负责执行工具调用的全流程（无需自己手动写解析逻辑）            |
| 5.整合回复  | chat_bot     | 模型基于真实数据生成人话           |

  - 工具调用示例代码：  

In [ ]:
# 1.定义工具：用 @tool 装饰器注册函数为工具，供 LLM 调用
@tool  # 核心装饰器：注册后内部会自动生成工具描述，供LLM判断是否调用
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    # 模拟天气数据
    weather_data = {
        "北京": "晴，温度 25°C，湿度 40%",
        "上海": "多云，温度 28°C，湿度 65%",
        "广州": "小雨，温度 30°C，湿度 80%",
        "深圳": "晴，温度 32°C，湿度 70%",
        "杭州": "阴，温度 26°C，湿度 55%",
    }
    return weather_data.get(city, f"抱歉，未找到城市 '{city}' 的天气信息。支持的城市：{', '.join(weather_data.keys())}")

# 设置工具列表：可添加多个调用的工具（如计算器、联网搜索等）
tools = [get_weather]

# 2.绑定工具：告诉模型现在有这些技能，利用LangChain的工具绑定方法 bind_tools，让LLM能识别工具并生成工具调用指令
llm_with_tools = ll.bind_tools(tools)

# 3.定义路由判断节点（too_router）：判断LLM是否生成了工具调用指令，如果有则路由到工具执行节点，否则继续正常对话流程
def tool_router(state: AgentState) -> Literal["tools", "end"]:
    # 获取最新一条消息（LLM的思考结果）
    last_message = state["messages"][-1]
    # 关键判断：如果最新消息包含tool_calls（工具调用指令），则去工具节点 tool_node 执行工具调用
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    # 否则结束
    return "end"

# 4.定义工具执行节点：接收工具调用指令，执行对应工具，并将结果返回给LLM
# ToolNode 是内置预构建节点，功能：解析tool_calls → 调用对应工具 → 生成ToolMessage
tool_node = ToolNode(tools)

**概念要点注意：**
  - ToolNode 内置节点，逻辑：  
  (1) 解析 LLM 的tool_calls指令（提取工具名、参数）  
  (2) 调用对应的工具函数（如get_weather("深圳今明两天天气")）  
  (3) 将工具返回结果包装为ToolMessage（LangChain 的工具消息类型）  
  (4) 自动将 ToolMessage 添加到对话历史，供后续节点使用  
  ToolNode 优势：无需关心工具调用的细节（如参数解析、异常处理），LangGraph 已封装好

**带工具调用的机器人**（完整实现，可运行）：

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. 加载环境变量
load_dotenv()

# 2. 创建支持工具调用的 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,
)

# ======================
# 3. 定义工具
# ======================
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    # 模拟天气数据
    weather_data = {
        "北京": "晴，温度 25°C，湿度 40%",
        "上海": "多云，温度 28°C，湿度 65%",
        "广州": "小雨，温度 30°C，湿度 80%",
        "深圳": "晴，温度 32°C，湿度 70%",
        "杭州": "阴，温度 26°C，湿度 55%",
    }
    return weather_data.get(city, f"抱歉，未找到城市 '{city}' 的天气信息。支持的城市：{', '.join(weather_data.keys())}")

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。例如：2 + 2, 10 * 5, 100 / 4"""
    try:
        # 简单的安全计算
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

@tool
def search_web(query: str) -> str:
    """模拟联网搜索（返回示例结果）"""
    return f"搜索结果：关于 '{query}'，这是一个模拟的搜索结果。在实际应用中，这里会返回真实的网络搜索内容。"

# 工具列表
tools = [calculator, get_weather, search_web]
# 将工具绑定到 LLM
llm_with_tools = llm.bind_tools(tools)

# ======================
# 4. 定义状态
# ======================
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ======================
# 5. 定义节点
# ======================
def chatbot(state: AgentState):
    """聊天机器人节点：根据消息决定是直接回复还是调用工具"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_router(state: AgentState) -> Literal["tools", "end"]:
    """定义路由判断节点（too_router）"""
    # 获取最新一条消息（LLM的思考结果）
    last_message = state["messages"][-1]
    # 关键判断：如果最新消息包含tool_calls（工具调用指令），则去工具节点 tool_node 执行工具调用
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    # 否则结束
    return "end"

# 定义工具执行节点：接收工具调用指令，执行对应工具，并将结果返回给LLM
tool_node = ToolNode(tools)

# ======================
# 6. 构建图
# ======================
graph = StateGraph(AgentState)

# 添加节点
graph.add_node("chatbot", chatbot)
graph.add_node("tools", ToolNode(tools))

# 设置入口
graph.set_entry_point("chatbot")

# 添加条件边：根据是否需要调用工具进行路由
graph.add_conditional_edges(
    "chatbot",
    tool_router,
    {
        "tools": "tools",  # 如果需要调用工具，前往 tools 节点
        "end": END,        # 否则结束
    }
)
# tools 节点执行完后，回到 chatbot 节点（让 AI 总结工具返回的结果）
graph.add_edge("tools", "chatbot")

# 编译图
app = graph.compile()

# ======================
# 7. 运行函数
# ======================
def run_chat(user_input: str):
    """运行聊天机器人，支持工具调用"""
    print("\n" + "="*50)
    print(f"User: {user_input}")
    print("Assistant:", end=" ", flush=True)

    # 初始化消息历史
    messages = [HumanMessage(content=user_input)]

    for msg, metadata in app.stream(
        {"messages": messages},
        stream_mode="messages",
    ):
        if isinstance(msg, AIMessage) and msg.content:
            print(msg.content, end="", flush=True)
        elif isinstance(msg, ToolMessage):
            print(f"\n[工具调用结果]: {msg.content[:100]}..." if len(msg.content) > 100 else f"\n[工具调用结果]: {msg.content}")
    
    print()

# ======================
# 8. 测试
# ======================
# 可视化显示 graph（需要在 Jupyter 环境中运行）
try:
    from IPython.display import Image, display
    print("\n=== Graph 可视化 ===")
    display(Image(app.get_graph().draw_mermaid_png()))
except ImportError:
    print("\n[提示] IPython 未安装，无法显示可视化图片。请在 Jupyter 环境中运行，或安装 ipython 包")

# 测试 1: 普通对话
print("\n=== 测试 1: 普通对话 ===")
run_chat("你好，请介绍一下你自己")

# 测试 2: 调用天气工具
print("\n=== 测试 3: 天气查询工具 ===")
run_chat("北京今天的天气怎么样？")

# 测试 3: 调用计算器工具
print("\n=== 测试 2: 计算器工具 ===")
run_chat("帮我计算 25 * 4 + 100")

# 测试 4: 调用搜索工具
print("\n=== 测试 4: 网络搜索工具 ===")
run_chat("搜索一下人工智能的最新进展")

#### 3.4.3 带记忆增强&工具调用的机器人
  - 核心流程：读历史 → 推理 → 工具调用 → 保存状态，即一次完整带记忆的交互。
  - 需使用 LangGraph 的 checkpoint 功能添加记忆功能。  
  - 具体分为 4 步：  
    1. 添加 MemorySaver 导入 —— `from langgraph.checkpoint.memory import MemorySaver`  
    2. 创建检查点实例 —— `memory = MemorySaver()`（仅存在内存中，程序重启后状态会丢失，适用于开发测试） 
    3. 使用检查点编译图图—— `app = graph.compile(checkpoint=memory)` 
    4. 通过添加 thread_id 参数配置对话ID —— 向 `app.stream(...,config=config,...)` 传递 `config={"configurable": {"thread_id": "your_thread_id"}}` 实现每个线程的持久化对话状态。
        - config 是「流程配置的统一入口」；
        - configurable 字段用于传递「可配置的全局参数」。  


In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. 加载环境变量
load_dotenv()

# 2. 创建支持工具调用的 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,
)

# 3. 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    # 模拟天气数据
    weather_data = {
        "北京": "晴，温度 25°C，湿度 40%",
        "上海": "多云，温度 28°C，湿度 65%",
        "广州": "小雨，温度 30°C，湿度 80%",
        "深圳": "晴，温度 32°C，湿度 70%",
        "杭州": "阴，温度 26°C，湿度 55%",
    }
    return weather_data.get(city, f"抱歉，未找到城市 '{city}' 的天气信息。支持的城市：{', '.join(weather_data.keys())}")

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。例如：2 + 2, 10 * 5, 100 / 4"""
    try:
        # 简单的安全计算
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

@tool
def search_web(query: str) -> str:
    """模拟联网搜索（返回示例结果）"""
    return f"搜索结果：关于 '{query}'，这是一个模拟的搜索结果。在实际应用中，这里会返回真实的网络搜索内容。"

# 工具列表
tools = [calculator, get_weather, search_web]
# 将工具绑定到 LLM
llm_with_tools = llm.bind_tools(tools)

# 4. 定义状态
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 5. 定义节点
def chatbot(state: AgentState):
    """聊天机器人节点：根据消息决定是直接回复还是调用工具"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_router(state: AgentState) -> Literal["tools", "end"]:
    """定义路由判断节点（too_router）"""
    # 获取最新一条消息（LLM的思考结果）
    last_message = state["messages"][-1]
    # 关键判断：如果最新消息包含tool_calls（工具调用指令），则去工具节点 tool_node 执行工具调用
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    # 否则结束
    return "end"

# 定义工具执行节点：接收工具调用指令，执行对应工具，并将结果返回给LLM
tool_node = ToolNode(tools)

# 6. 构建图
graph = StateGraph(AgentState)

# 添加节点
graph.add_node("chatbot", chatbot)
graph.add_node("tools", ToolNode(tools))

# 设置入口
graph.set_entry_point("chatbot")

# 添加条件边：根据是否需要调用工具进行路由
graph.add_conditional_edges(
    "chatbot",
    tool_router,
    {
        "tools": "tools",  # 如果需要调用工具，前往 tools 节点
        "end": END,        # 否则结束
    }
)
# tools 节点执行完后，回到 chatbot 节点（让 AI 总结工具返回的结果）
graph.add_edge("tools", "chatbot")

# ======================
# 编译图（启用检查点以支持记忆功能）
# ======================
memory = MemorySaver()
app = graph.compile(checkpointer=memory)


# 7. 运行函数
def run_chat(user_input: str, thread_id: str = "default"):
    """运行聊天机器人，支持工具调用和对话历史记忆"""
    print("\n" + "="*50)
    print(f"User: {user_input}")
    print("Assistant:", end=" ", flush=True)

    # 初始化消息历史（空列表，检查点会自动加载历史）
    messages = []
    # 通过添加 thread_id 参数配置对话ID
    config = {"configurable": {"thread_id": thread_id}}
    for msg, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages",
    ):
        if isinstance(msg, AIMessage) and msg.content:
            print(msg.content, end="", flush=True)
        elif isinstance(msg, ToolMessage):
            print(f"\n[工具调用结果]: {msg.content[:100]}..." if len(msg.content) > 100 else f"\n[工具调用结果]: {msg.content}")

    print()

# 8. 测试
# 可视化显示 graph（需要在 Jupyter 环境中运行）
try:
    from IPython.display import Image, display
    print("\n=== Graph 可视化 ===")
    display(Image(app.get_graph().draw_mermaid_png()))
except ImportError:
    print("\n[提示] IPython 未安装，无法显示可视化图片。请在 Jupyter 环境中运行，或安装 ipython 包")


# 测试 多轮对话（测试记忆功能）
print("\n=== 测试: 多轮对话（记忆功能测试）===")
memory_thread = "memory_test_001"

print("\n--- 第 1 轮：自我介绍 ---")
run_chat("你好，我叫小明，我是一名程序员。", thread_id=memory_thread)

print("\n--- 第 2 轮：询问名字（测试是否记住）---")
run_chat("你还记得我叫什么名字吗？", thread_id=memory_thread)

print("\n--- 第 3 轮：询问职业（测试是否记住）---")
run_chat("我是做什么的？", thread_id=memory_thread)

print("\n--- 第 4 轮：新话题（不同 thread_id，无记忆）---")
run_chat("你还记得我叫什么名字吗？", thread_id="new_thread_002")


**概念要点注意：**
  - 1. MemorySaver 是内存持久化，跑个demo还可以，若需要**本地/生产环境持久化**则需要使用官方数据库版检查点保存器：  

| 存储后端                         | Saver 类                                              | 安装包                                      |
|----------------------------------|-------------------------------------------------------|---------------------------------------------|
| 内存                             | `MemorySaver`                                         | `langgraph`（内置）                         |
| SQLite                          | `SqliteSaver`, `AsyncSqliteSaver`                     | `langgraph-checkpoint-sqlite`               |
| PostgreSQL                      | `PostgresSaver`, `AsyncPostgresSaver`                 | `langgraph-checkpoint-postgres`             |
| MongoDB                         | `MongoDBSaver`, `AsyncMongoDBSaver`                   | `langgraph-checkpoint-mongodb`              |
| Redis                           | `RedisSaver`, `AsyncRedisSaver`                       | `langgraph-checkpoint-redis`                |
| AWS DynamoDB                    | `DynamoDBSaver`, `AsyncDynamoDBSaver`                 | `langgraph-checkpoint-aws`                  |
| Google Cloud Spanner            | `SpannerSaver`, `AsyncSpannerSaver`                   | `langgraph-checkpoint-spanner`              |
| Apache Cassandra / DataStax Astra DB | `CassandraSaver`, `AsyncCassandraSaver`         | `langgraph-checkpoint-cassandra`            |
| 浏览器 IndexedDB                | `IndexedDBSaver`                                      | `@langchain/langgraph-checkpoint-web`       |  

**几个常用场景推荐：**  
- **生产环境首选**：`PostgreSQL` / `MongoDB` — 成熟、稳定、支持并发  
- **云原生 / Serverless**：`DynamoDB` — 适合 AWS 环境  
- **缓存 / 高性能**：`Redis` — 适合需要低延迟的场景  
- **快速原型**：`SqliteSaver` 或 `MemorySaver` — 轻量，无需额外服务  
- **多节点部署**：避免使用 `MemorySaver`（仅进程内有效），改用任何外部数据库版 Saver  

  > 所有外部 Saver 都需要单独安装对应包，例
  > ```bash
  > pip install langgraph-checkpoint-mongodb  

**生产环境持久化机器人示例：**  
  - 使用前准备  
    1. 安装依赖
        ```bash
        pip install langgraph-checkpoint-postgres
        ````
    2. 配置环境变量  
        `.env` 填入真实配置  
    3. 初始化数据库（首次运行）  
        ```bash
        python setup_postgres.py
        ```
  - 运行主程序：  

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. 加载环境变量
load_dotenv()

# ======================
# 2. 构建 PostgreSQL DSN
# ======================
PG_DSN = os.getenv("PG_DSN")
if not PG_DSN:
    PG_DSN = (
        f"postgresql://{os.getenv('PG_USER', 'postgres')}:{os.getenv('PG_PASSWORD', '')}"
        f"@{os.getenv('PG_HOST', 'localhost')}:{os.getenv('PG_PORT', '5432')}"
        f"/{os.getenv('PG_DBNAME', 'langgraph_db')}"
    )

# 3. 创建支持工具调用的 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,
)

# 4. 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    weather_data = {
        "北京": "晴，温度 25°C，湿度 40%",
        "上海": "多云，温度 28°C，湿度 65%",
        "广州": "小雨，温度 30°C，湿度 80%",
        "深圳": "晴，温度 32°C，湿度 70%",
        "杭州": "阴，温度 26°C，湿度 55%",
    }
    return weather_data.get(city, f"抱歉，未找到城市 '{city}' 的天气信息。支持的城市：{', '.join(weather_data.keys())}")

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。例如：2 + 2, 10 * 5, 100 / 4"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

@tool
def search_web(query: str) -> str:
    """模拟联网搜索（返回示例结果）"""
    return f"搜索结果：关于 '{query}'，这是一个模拟的搜索结果。在实际应用中，这里会返回真实的网络搜索内容。"

tools = [calculator, get_weather, search_web]
llm_with_tools = llm.bind_tools(tools)

# 5. 定义状态、节点、图
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: AgentState):
    """聊天机器人节点"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_router(state: AgentState) -> Literal["tools", "end"]:
    """路由判断节点"""
    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    return "end"

# 构建图
graph = StateGraph(AgentState)
graph.add_node("chatbot", chatbot)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("chatbot")
graph.add_conditional_edges("chatbot", tool_router, {"tools": "tools", "end": END})
graph.add_edge("tools", "chatbot")

# 6. 运行函数
def _format_tool_content(content: str) -> str:
    """格式化工具调用结果的显示"""
    return content[:100] + "..." if len(content) > 100 else content

def run_chat(user_input: str, thread_id: str = "default", app=None):
    """运行聊天机器人，支持工具调用和 PostgreSQL 持久化记忆"""
    if app is None:
        raise ValueError("app 参数不能为 None，请先创建编译后的图")

    print("\n" + "="*50)
    print(f"User: {user_input}")
    print("Assistant:", end=" ", flush=True)

    config = {"configurable": {"thread_id": thread_id}}

    for msg, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages",
    ):
        if isinstance(msg, AIMessage) and msg.content:
            print(msg.content, end="", flush=True)
        elif isinstance(msg, ToolMessage):
            print(f"\n[工具调用结果]: {_format_tool_content(msg.content)}")

    print()


# 7. 测试
try:
    from IPython.display import Image, display
    print("\n=== Graph 可视化 ===")
    display(Image(app.get_graph().draw_mermaid_png()))
except ImportError:
    print("\n[提示] IPython 未安装，无法显示可视化图片。请在 Jupyter 环境中运行，或安装 ipython 包")

# ======================
# 使用上下文管理器确保 PostgreSQL 连接正确打开和关闭
# ======================
with PostgresSaver.from_conn_string(PG_DSN) as checkpointer:
    checkpointer.setup()
    app = graph.compile(checkpointer=checkpointer)

    print("\n=== 测试: 多轮对话（PostgreSQL 持久化记忆）===")
    memory_thread = "postgres_memory_001"

    print("\n--- 第 1 轮：自我介绍 ---")
    run_chat("你好，我叫小明，我是一名程序员。", thread_id=memory_thread, app=app)

    print("\n--- 第 2 轮：询问名字（测试是否记住）---")
    run_chat("你还记得我叫什么名字吗？", thread_id=memory_thread, app=app)

    print("\n--- 第 3 轮：询问职业（测试是否记住）---")
    run_chat("我是做什么的？", thread_id=memory_thread, app=app)

    print("\n--- 第 4 轮：新话题（不同 thread_id，无记忆）---")
    run_chat("你还记得我叫什么名字吗？", thread_id="new_thread_002", app=app)

    print("\n=== PostgreSQL 持久化记忆测试完成 ===")


#### 3.4.4 给机器人安装**监控**
  - 在构建复杂的 LLM 应用（尤其是涉及 RAG 或多 Agent 系统）时，可观测性（Observability）、评估（Evaluation）和提示管理（Prompt Management）是必不可少的环节。LangSmith 和 Langfuse 都是这一领域的佼佼者，但它们的设计哲学和主攻方向有着显著差异。
  - LangSmith vs. Langfuse：深度对比与选型总结

| 平台 | 类型 | 开闭源 | 适合性 | 定价（免费层级） |
|--------------|----------|----------------|----------------|----------------------------------|
| Langfuse | 追踪与提示管理 | 开源（MIT 核心；企业版提供 EE 功能） | 自托管部署、数据主权、框架无关设置 | 云端：每月 5 万单位 — 自托管：无限制 |
| LangSmith | 追踪与调试 | 闭源（专有、黑盒） | 原生工作流，图形状态可视化 | 每月 5 千次追踪 |

  - 总结：Langfuse 是 LangSmith 的替代品，它更适合于企业级应用，具有更丰富的功能和功能。


In [ ]:
# 第一步 — 安装依赖项：
! pip install langfuse

# 第二步 — 配置环境变量：(或将 .env.example中配置并复制到 .env)
"""
LANGFUSE_ENABLED=true.   # 是否启动追踪
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted URL
"""

In [ ]:
# 第三步：集成监控模块
from langfuse import get_client
from langfuse.langchain import CallbackHandler

class Observability:
    """Langfuse client + callback handler + 强制刷盘能力"""
    def __init__(self):
        self._client = get_client()  # 自动读取环境变量，用于发送trace/span/logs
        self._handler = CallbackHandler()
    
    def get_handler(self):
        return self._handler         # LangChain/LangGraph 的事件监听器，自动捕捉：LLM调用、Tool调用、Chain执行过程、Token usage、latency
    
    def flush(self):
        self._client.flush()         # 异步将追踪数据发送到 Langfuse 服务器

In [ ]:
# 第四步：给机器人安装 监控

import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. 加载环境变量
load_dotenv()

# 2. 创建支持工具调用的 LLM
llm = ChatOpenAI(
    model="qwen-plus-latest",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    streaming=True,
)

# 3. 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气（模拟数据）"""
    # 模拟天气数据
    weather_data = {
        "北京": "晴，温度 25°C，湿度 40%",
        "上海": "多云，温度 28°C，湿度 65%",
        "广州": "小雨，温度 30°C，湿度 80%",
        "深圳": "晴，温度 32°C，湿度 70%",
        "杭州": "阴，温度 26°C，湿度 55%",
    }
    return weather_data.get(city, f"抱歉，未找到城市 '{city}' 的天气信息。支持的城市：{', '.join(weather_data.keys())}")

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。例如：2 + 2, 10 * 5, 100 / 4"""
    try:
        # 简单的安全计算
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

@tool
def search_web(query: str) -> str:
    """模拟联网搜索（返回示例结果）"""
    return f"搜索结果：关于 '{query}'，这是一个模拟的搜索结果。在实际应用中，这里会返回真实的网络搜索内容。"

# 工具列表
tools = [calculator, get_weather, search_web]
# 将工具绑定到 LLM
llm_with_tools = llm.bind_tools(tools)

# 4. 定义状态
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 5. 定义节点
def chatbot(state: AgentState):
    """聊天机器人节点：根据消息决定是直接回复还是调用工具"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_router(state: AgentState) -> Literal["tools", "end"]:
    """定义路由判断节点（too_router）"""
    # 获取最新一条消息（LLM的思考结果）
    last_message = state["messages"][-1]
    # 关键判断：如果最新消息包含tool_calls（工具调用指令），则去工具节点 tool_node 执行工具调用
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    # 否则结束
    return "end"

# 定义工具执行节点：接收工具调用指令，执行对应工具，并将结果返回给LLM
tool_node = ToolNode(tools)

# 6. 构建图
graph = StateGraph(AgentState)

# 添加节点
graph.add_node("chatbot", chatbot)
graph.add_node("tools", ToolNode(tools))

# 设置入口
graph.set_entry_point("chatbot")

# 添加条件边：根据是否需要调用工具进行路由
graph.add_conditional_edges(
    "chatbot",
    tool_router,
    {
        "tools": "tools",  # 如果需要调用工具，前往 tools 节点
        "end": END,        # 否则结束
    }
)
# tools 节点执行完后，回到 chatbot 节点（让 AI 总结工具返回的结果）
graph.add_edge("tools", "chatbot")

# ======================
# 编译图（启用检查点以支持记忆功能）
# ======================
memory = MemorySaver()
app = graph.compile(checkpointer=memory)


# 7. 运行函数
def run_chat(user_input: str, thread_id: str = "default"):
    """运行聊天机器人，支持工具调用和对话历史记忆"""
    print("\n" + "="*50)
    print(f"User: {user_input}")
    print("Assistant:", end=" ", flush=True)

    # 通过添加 thread_id 参数配置对话ID
    config = {"configurable": {"thread_id": thread_id}}
    # ======================
    # 把 observability 注入整个图中：在入口挂一次 callback，全图自动零侵入埋点，全链路追踪
    # ======================
    observability = Observability()
    handler = observability.get_handler()
    if handler:
        config["callbacks"] = [handler]

    for msg, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages",
    ):
        if isinstance(msg, AIMessage) and msg.content:
            print(msg.content, end="", flush=True)
        elif isinstance(msg, ToolMessage):
            print(f"\n[工具调用结果]: {msg.content[:100]}..." if len(msg.content) > 100 else f"\n[工具调用结果]: {msg.content}")

    print()

    observability.flush()  # 确保未发送的跟踪数据已发送

# 8. 测试
# 可视化显示 graph（需要在 Jupyter 环境中运行）
try:
    from IPython.display import Image, display
    print("\n=== Graph 可视化 ===")
    display(Image(app.get_graph().draw_mermaid_png()))
except ImportError:
    print("\n[提示] IPython 未安装，无法显示可视化图片。请在 Jupyter 环境中运行，或安装 ipython 包")


# 测试 多轮对话（测试记忆功能）
print("\n=== 测试: 多轮对话（记忆功能测试）===")
memory_thread = "memory_test_001"

print("\n--- 第 1 轮：自我介绍 ---")
run_chat("你好，我叫小明，我是一名程序员。", thread_id=memory_thread)

print("\n--- 第 2 轮：询问名字（测试是否记住）---")
run_chat("你还记得我叫什么名字吗？", thread_id=memory_thread)

print("\n--- 第 3 轮：询问职业（测试是否记住）---")
run_chat("我是做什么的？", thread_id=memory_thread)

print("\n--- 第 4 轮：新话题（不同 thread_id，无记忆）---")
run_chat("你还记得我叫什么名字吗？", thread_id="new_thread_002")
